In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class FeatureStoreConfig:

    root_dir: Path

    feature_columns_path: Path
    
    feature_repo_path: Path

    offline_feature_path: Path

    registry_path: Path

    online_store_path: Path

    feature_store_yaml_path: Path

    project_name: str

    feature_view_name: str

    feature_service_name: str

    entity_name: str

    join_key: str

    timestamp_column: str

    start_date: str

    end_date: str

In [6]:
from src.ride_demand_forecasting_and_marketplace_optimization.constants import *
from src.ride_demand_forecasting_and_marketplace_optimization.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH,
                 feature_cfg_filepath = FEATURE_COLUMNS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        self.feature_cfg = read_yaml(feature_cfg_filepath)

        create_directories([self.config.artifacts_root])

    def get_feature_store_config(self):

        config = self.config.feature_store

        create_directories(
            [config.root_dir]
        )


        return FeatureStoreConfig(

            root_dir=Path(config.root_dir),
            
            feature_columns_path=Path(config.feature_columns_path),
            
            feature_repo_path=Path(config.feature_repo_path),

            offline_feature_path=Path(config.offline_feature_path),

            registry_path=Path(config.registry_path),

            online_store_path=Path(config.online_store_path),

            feature_store_yaml_path=Path(config.feature_store_yaml_path),

            project_name=config.project_name,

            feature_view_name=config.feature_view_name,

            feature_service_name=config.feature_service_name,

            entity_name=config.entity_name,

            join_key=config.join_key,

            timestamp_column=config.timestamp_column,

            start_date=config.start_date,

            end_date=config.end_date
        
        )    

In [8]:
import shutil
import subprocess
import sys
from pathlib import Path
import time
from src.ride_demand_forecasting_and_marketplace_optimization import logger

In [9]:
class FeatureStoreComponent:

    def __init__(self, config: FeatureStoreConfig):
        self.config = config

    def create_repo(self):

        repo_path = self.config.feature_repo_path

        repo_path.mkdir(
            parents=True,
            exist_ok=True
        )

        # remove old files
        for file in repo_path.glob("*.py"):
            file.unlink()

        (repo_path / "__init__.py").write_text("")
        shutil.copy(self.config.feature_columns_path, repo_path / "feature_columns.yaml")

        # -------------------------------------------------
        # feature_store.yaml
        # -------------------------------------------------

        yaml_content = f"""
project: {self.config.project_name}

registry: registry.db

provider: local
entity_key_serialization_version: 2
online_store:
    type: sqlite
    path: online_store.db
"""

        (repo_path / "feature_store.yaml").write_text(
            yaml_content.strip()
        )

        # -------------------------------------------------
        # entities.py
        # -------------------------------------------------

        entities_content = f"""
from feast import Entity, ValueType

zone_timestamp = Entity(
    name="{self.config.entity_name}",
    join_keys=["{self.config.join_key}"],
    value_type=ValueType.STRING
)
"""

        (repo_path / "entities.py").write_text(
            entities_content.strip()
        )

        # -------------------------------------------------
        # data_sources.py
        # -------------------------------------------------

        datasource_content = f'''
from feast import FileSource

forecast_source = FileSource(
    path=r"{self.config.offline_feature_path.resolve()}",
    event_timestamp_column="{self.config.timestamp_column}",
)
'''

        (repo_path / "data_sources.py").write_text(
            datasource_content.strip()
        )

        # -------------------------------------------------
        # feature_views.py
        # -------------------------------------------------

        feature_view_content = f"""
from feast import FeatureView, Field
from feast.types import (
    Int32,
    Int64,
    Float32,
    Float64,
    String,
)

from entities import zone_timestamp
from data_sources import forecast_source

import yaml
from pathlib import Path
from datetime import timedelta


TYPE_MAPPING = {{
    "Int32": Int32,
    "Int64": Int64,
    "Float32": Float32,
    "Float64": Float64,
    "String": String,
    "object": String,
}}

def load_feature_config():

    config_path = (
        Path(__file__).parent
        / "feature_columns.yaml"
    )

    with open(config_path, "r") as file:
        return yaml.safe_load(file)


def build_schema():

    config = load_feature_config()

    feature_cfg = config["feature_columns"]

    feast_features = feature_cfg["feast_features"]

    feature_dtypes = feature_cfg["feature_dtypes"]

    schema = []

    for feature_name in feast_features:

        dtype_name = feature_dtypes[feature_name]

        schema.append(
            Field(
                name=feature_name,
                dtype=TYPE_MAPPING[dtype_name],
            )
        )

    return schema


ride_feature_view = FeatureView(
    name="{self.config.feature_view_name}",
    entities=[zone_timestamp],
    ttl=timedelta(days=30),
    schema=build_schema(),
    source=forecast_source,
)
"""

        (repo_path / "feature_views.py").write_text(
            feature_view_content.strip()
        )

        # -------------------------------------------------
        # feature_services.py
        # -------------------------------------------------

        service_content = f"""
from feast import FeatureService

from feature_views import ride_feature_view


forecast_service = FeatureService(

    name="{self.config.feature_service_name}",

    features=[
        ride_feature_view
    ]
)
"""

        (repo_path / "feature_services.py").write_text(
            service_content.strip()
        )

    def clean_registry(self):

        registry = self.config.registry_path

        online_store = self.config.online_store_path

        if registry.exists():
            registry.unlink()

        if online_store.exists():
            online_store.unlink()

    def apply(self):
        
        feature_file = Path(self.config.offline_feature_path)

        if not feature_file.exists():
            raise FileNotFoundError(
                f"Offline feature file not found: {feature_file}"
            )

        result = subprocess.run(

            [
                sys.executable,
                "-m",
                "feast.cli",
                "apply"
            ],

            cwd=str(
                self.config.feature_repo_path
            ),

            capture_output=True,

            text=True
        )

        print(result.stdout)

        if result.returncode != 0:

            print(result.stderr)

            raise Exception(
                f"Feast Apply Failed\\n{result.stderr}"
            )
        
    def _materialize_features(self):
        """
        Materialize features from offline store to online store.
        """

        result = subprocess.run(

            [
                sys.executable,
                "-m",
                "feast.cli",
                "materialize",
                str(self.config.start_date),
                str(self.config.end_date)
            ],

            cwd=str(
                self.config.feature_repo_path
            ),

            capture_output=True,

            text=True
        )

        print("=" * 100)
        print("MATERIALIZATION")
        print("=" * 100)

        print("STDOUT:")
        print(result.stdout)

        print("=" * 100)

        print("STDERR:")
        print(result.stderr)

        print("=" * 100)

        if result.returncode != 0:

            raise Exception(
                f"Feast Materialization Failed:\n{result.stderr}"
            )

        logger.info(
            "Features materialized successfully"
        )    

    def initiate_feature_store(self):

        self.create_repo()

        self.clean_registry()

        start = time.time()
        self.apply()
        print(f"Apply Time: {time.time()-start:.2f}s")

        #start = time.time()
        #self._materialize_features()
        #print(f"Materialize Time: {time.time()-start:.2f}s")

        logger.info(
            "Feature Store Component completed successfully"
        )

In [10]:
try:
    config = ConfigurationManager()

    feature_store_config = (
        config.get_feature_store_config()
    )

    feature_store = FeatureStoreComponent(
        config= feature_store_config
    )


    feature_store.initiate_feature_store()
except Exception as e:
    logger.exception(e)
    raise e    

[2026-07-16 13:59:57,788: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-16 13:59:57,792: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-16 13:59:57,822: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-07-16 13:59:57,856: INFO: common: yaml file: config\feature_columns.yaml loaded successfully]
[2026-07-16 13:59:57,866: INFO: common: created directory at: artifacts]
[2026-07-16 13:59:57,872: INFO: common: created directory at: artifacts/feature_store]
No project found in the repository. Using project name ride_marketplace_forecasting defined in feature_store.yaml
Applying changes for project ride_marketplace_forecasting
Created project ride_marketplace_forecasting
Created entity zone_timestamp
Created feature view ride_marketplace_features
Created feature service forecast_service

Created sqlite table ride_marketplace_forecasting_ride_marketplace_features


Apply Time: 18.11s
[2026-07-16 14:00:16,120: INFO: 197344501

In [11]:
print(feature_store_config.offline_feature_path)

artifacts\feature_engineering\featured_data.parquet


In [12]:
from pathlib import Path

file_path = Path(
    "artifacts/feature_store/offline_features.parquet"
)

print(file_path.exists())

False


In [14]:
from feast import FeatureStore

store = FeatureStore(repo_path="src/ride_demand_forecasting_and_marketplace_optimization/feature_repo")

print(store.list_entities())
print(store.list_feature_views())
print(store.list_feature_services())

[2026-07-16 14:01:31,980: INFO: registry: Registry cache expired, so refreshing]
[Entity(
    name='zone_timestamp',
    value_type=<ValueType.STRING: 2>,
    join_key='zone_timestamp_key',
    description='',
    tags={},
    owner='',
    created_timestamp=datetime.datetime(2026, 7, 16, 8, 30, 15, 171326),
    last_updated_timestamp=datetime.datetime(2026, 7, 16, 8, 30, 15, 171326)
)]
[2026-07-16 14:01:31,984: INFO: registry: Registry cache expired, so refreshing]


c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\pydantic\_internal\_fields.py:186: UserWarning: Field name "vector_enabled" shadows an attribute in parent "VectorStoreConfig"; 
  warnings.warn(
c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\pydantic\_internal\_fields.py:186: UserWarning: Field name "vector_len" shadows an attribute in parent "VectorStoreConfig"; 
  warnings.warn(
c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\pydantic\_internal\_fields.py:186: UserWarning: Field name "similarity" shadows an attribute in parent "VectorStoreConfig"; 
  warnings.warn(
c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\feast\entity.py:173: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  entity = cls(
c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\feast\entity.py:173: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'zone_tim

[<FeatureView(name = ride_marketplace_features, entities = ['zone_timestamp'], ttl = 30 days, 0:00:00, stream_source = None, batch_source = {
  "type": "BATCH_FILE",
  "timestampField": "timestamp",
  "fileOptions": {
    "uri": "D:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization\\artifacts\\feature_engineering\\featured_data.parquet"
  },
  "name": "D:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization\\artifacts\\feature_engineering\\featured_data.parquet"
}, entity_columns = [Field(
    name='zone_timestamp_key',
    dtype=<PrimitiveFeastType.STRING: 2>,
    description='',
    tags={}
    vector_index=False
    vector_search_metric=''
)], features = [Field(
    name='zone_id',
    dtype=<PrimitiveFeastType.INT32: 3>,
    description='',
    tags={}
    vector_index=False
    vector_search_metric=''
), Field(
    name='zone_type',
    dtype=<PrimitiveFeastType.STRING: 2>,
    description='',
    tags={}
    vector_index=Fa